# Leaderboard Pipeline 02 — Ranked Manual Label Review

This notebook creates a small, prioritized review queue from OOF predictions. It does **not** automatically relabel images.

Review order emphasizes extreme model–label disagreement, cross-label duplicates, duplicate candidates, and intrinsically ambiguous images.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts"))
DATA_ROOT = REPO_ROOT / "BDC2026"
MODEL_OUTPUT = REPO_ROOT / "outputs_dinov3_lora_384"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
DISPLAY_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MODEL_OUTPUT:", MODEL_OUTPUT)
print("PIPELINE_ROOT:", PIPELINE_ROOT)


In [ ]:
import numpy as np
import pandas as pd

from error_analysis import (
    load_oof, enrich_predictions, add_duplicate_flags, add_error_categories, show_image_grid,
)

STAGE_DIR = PIPELINE_ROOT / "02_label_review"
GRIDS_DIR = STAGE_DIR / "review_grids"
GRIDS_DIR.mkdir(parents=True, exist_ok=True)

EDA_DIRS = [
    REPO_ROOT / "eda_outputs",
    REPO_ROOT / "eda_outputs_dino",
    REPO_ROOT / "eda_outputs" / "notebook_pipeline" / "02_exact_duplicates",
    REPO_ROOT / "eda_outputs" / "notebook_pipeline" / "03_phash",
    REPO_ROOT / "eda_outputs" / "notebook_pipeline" / "04_dino_embeddings",
]

oof = enrich_predictions(load_oof(MODEL_OUTPUT, DATA_ROOT))
oof = add_duplicate_flags(oof, EDA_DIRS)
oof = add_error_categories(oof, high_confidence=.85, very_high_confidence=.95, low_margin=.15)
wrong = oof[~oof["correct"]].copy()
print("Wrong OOF predictions:", len(wrong))


In [ ]:
wrong["priority_score"] = wrong["confidence"] * (1-wrong["true_probability"]) * np.maximum(wrong["margin"], 1e-6)
wrong["review_tier"] = np.select(
    [
        (wrong["confidence"] >= .99) & (wrong["true_probability"] <= .01),
        (wrong["confidence"] >= .97) & (wrong["true_probability"] <= .03),
        (wrong["confidence"] >= .95) & (wrong["true_probability"] <= .05),
        wrong["is_cross_label_duplicate"],
        wrong["is_duplicate_candidate"],
        wrong["margin"] <= .15,
    ],
    [
        "A_extreme_model_label_disagreement",
        "B_very_strong_model_label_disagreement",
        "C_strong_model_label_disagreement",
        "D_cross_label_duplicate",
        "E_duplicate_or_near_duplicate",
        "F_intrinsic_ambiguity",
    ],
    default="G_general_error",
)
order = {name:i for i,name in enumerate([
    "A_extreme_model_label_disagreement", "B_very_strong_model_label_disagreement",
    "C_strong_model_label_disagreement", "D_cross_label_duplicate",
    "E_duplicate_or_near_duplicate", "F_intrinsic_ambiguity", "G_general_error",
])}
wrong["tier_order"] = wrong["review_tier"].map(order)
wrong = wrong.sort_values(["tier_order", "priority_score", "confidence"], ascending=[True,False,False]).reset_index(drop=True)
wrong[["review_tier","confidence","true_probability","margin","true_name","pred_name","resolved_path"]].head(30)


In [ ]:
tier_summary = wrong.groupby("review_tier", as_index=False).agg(
    images=("path","size"),
    mean_confidence=("confidence","mean"),
    mean_true_probability=("true_probability","mean"),
    mean_priority=("priority_score","mean"),
)
display(tier_summary)


In [ ]:
show_image_grid(
    wrong, n=30, sort_by="priority_score", ascending=False,
    title="Highest-priority manual label review",
    save_path=GRIDS_DIR / "highest_priority_review.png",
)


In [ ]:
for tier in list(order)[:6]:
    subset = wrong[wrong["review_tier"] == tier]
    if len(subset):
        show_image_grid(
            subset, n=20, sort_by="priority_score", ascending=False,
            title=tier, save_path=GRIDS_DIR / f"{tier}.png",
        )


In [ ]:
review_columns = [
    "path","resolved_path","class_name","label","true_name","oof_pred","pred_name","fold",
    "confidence","true_probability","margin","normalized_entropy","priority_score","review_tier",
    "error_category","duplicate_sources","is_cross_label_duplicate",
]
queue = wrong[review_columns].copy()
queue["review_action"] = ""
queue["new_label"] = ""
queue["review_reason"] = ""
queue["reviewer"] = ""
queue["second_review_required"] = False

queue.to_csv(STAGE_DIR / "ranked_review_queue.csv", index=False)
queue.head(300).to_csv(STAGE_DIR / "priority_review_first_300.csv", index=False)
queue[queue["confidence"] >= .95].to_csv(STAGE_DIR / "high_confidence_label_review.csv", index=False)
tier_summary.to_csv(STAGE_DIR / "review_tier_summary.csv", index=False)
print("Saved:", STAGE_DIR)


## Review policy

Use one of four decisions:

- `keep`: original label is valid.
- `relabel`: visually clear under the competition taxonomy; fill `new_label` and a reason.
- `exclude`: corrupt, unusable, irreducibly ambiguous, or outside the intended domain.
- `needs_second_review`: uncertain taxonomy or mixed-material case.

A confident OOF disagreement is evidence, not proof. Prioritize obvious electronics with non-electronic labels, but be conservative with packaging-versus-contents and mixed-material products.
